# 04 · Experiment comparison and confusion analysis

Reads `outputs/metrics/suite_runs.csv` produced by `make experiments` (subject-independent KARD evaluation, train s01-s06 / val s07-s08 / test s09-s10). Nothing is shown for experiments that have not been run.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from skeleton_action.constants import CLASS_NAMES, JOINT_NAMES, KARD_EDGES, J
from skeleton_action.data.kard import KardCollection
INTERIM = ROOT / "data/interim/kard_realworld.npz"
assert INTERIM.exists(), "run `make data` first"
col = KardCollection.load(INTERIM)
print(len(col), "sequences")

In [ ]:
runs_path = ROOT / "outputs/metrics/suite_runs.csv"
runs = pd.read_csv(runs_path) if runs_path.exists() else None
if runs is None:
    print("No experiment results yet: run `make experiments` (or `make aggregate` after partial runs).")
else:
    summ = runs.groupby("experiment").agg(n=("seed", "size"), val_f1=("val_macro_f1", "mean"),
        test_acc=("test_accuracy", "mean"), test_acc_sd=("test_accuracy", "std"),
        test_f1=("test_macro_f1", "mean"), test_f1_sd=("test_macro_f1", "std"),
        params=("params", "first"), cpu1_ms=("cpu1_ms", "mean"))
    display(summ.round(4))
    summ.test_f1.plot.bar(yerr=summ.test_f1_sd, capsize=3, figsize=(9, 3.5), title="test macro-F1 (mean ± sd over seeds)")
    plt.ylim(0, 1); plt.show()

## Confusion analysis of the main ST-GCN

In [ ]:
def load_run(name, seed):
    m = ROOT / "outputs/metrics" / name / f"seed{seed}"
    if not (m / "summary.json").exists():
        return None
    out = {"summary": json.loads((m / "summary.json").read_text()),
           "history": pd.read_csv(m / "history.csv")}
    for part in ("val", "test"):
        p = m / f"{part}_metrics.json"
        if p.exists():
            out[part] = json.loads(p.read_text())
    return out
cands = [("e3_stgcn_xyz", s) for s in (42, 43, 44)] + [("smoke_stgcn", 0)]
run = next(((n, s, r) for n, s in cands if (r := load_run(n, s))), None)
if run:
    name, seed, r = run
    print(name, seed, "(SMOKE RUN - not a result)" if name.startswith("smoke") else "")
    cm = np.array(r["test"]["confusion_matrix"])
    norm = cm / cm.sum(1, keepdims=True)
    plt.figure(figsize=(9, 8)); plt.imshow(norm, cmap="Blues"); plt.colorbar()
    plt.xticks(range(18), CLASS_NAMES, rotation=70, ha="right", fontsize=7); plt.yticks(range(18), CLASS_NAMES, fontsize=7)
    plt.title("test confusion (row-normalised)"); plt.tight_layout(); plt.show()
    display(pd.DataFrame(r["test"]["top_confusions"]))

## Are confusions caused by similar poses or similar dynamics?

For each confused pair we compare class templates computed on the *training* subjects: (a) **pose distance** = distance between time-averaged normalised poses, (b) **dynamics distance** = distance between mean velocity-magnitude profiles. Small pose distance + larger dynamics distance suggests the model under-uses temporal information; small values for both suggest the classes are intrinsically hard to separate with 15 joints.

In [ ]:
from skeleton_action.preprocessing.transforms import PreprocessConfig, preprocess_sequence
pre = PreprocessConfig(velocity=True)
train_idx = np.flatnonzero(np.isin(col.subjects, [1, 2, 3, 4, 5, 6]))
X = np.stack([preprocess_sequence(col.sequence(i), pre) for i in train_idx]); y = col.labels[train_idx]
pose_t = np.stack([X[y == c, :, :, :3].mean(axis=(0, 1)) for c in range(18)])           # [18, 15, 3]
dyn_t = np.stack([np.linalg.norm(X[y == c, :, :, 3:], axis=-1).mean(0) for c in range(18)])  # [18, T, 15]
def pose_d(a, b): return float(np.linalg.norm(pose_t[a] - pose_t[b], axis=-1).mean())
def dyn_d(a, b): return float(np.abs(dyn_t[a] - dyn_t[b]).mean())
allp = [pose_d(a, b) for a in range(18) for b in range(18) if a < b]
alld = [dyn_d(a, b) for a in range(18) for b in range(18) if a < b]
print("median over all pairs: pose %.3f dynamics %.4f" % (np.median(allp), np.median(alld)))
if run:
    rows = []
    for c in r["test"]["top_confusions"]:
        a, b = CLASS_NAMES.index(c["true"]), CLASS_NAMES.index(c["pred"])
        rows.append({**c, "pose_dist": pose_d(a, b), "pose_pct": np.mean(np.array(allp) < pose_d(a, b)),
                     "dyn_dist": dyn_d(a, b), "dyn_pct": np.mean(np.array(alld) < dyn_d(a, b))})
    display(pd.DataFrame(rows).round(3))

`*_pct` = percentile of the pair among all 153 class pairs (low = more similar). Interpretation must be written from the actual numbers after `make experiments`.